## 1. 环境自检

In [ ]:
import sys
from pathlib import Path

# ---------- 动态推导项目根目录 ----------
def find_project_root(marker: str = "AGENTS.md", max_depth: int = 5) -> Path:
    """从当前工作目录向上查找包含标记文件的目录作为项目根。"""
    current = Path.cwd()
    for _ in range(max_depth):
        if (current / marker).is_file():
            return current
        parent = current.parent
        if parent == current:
            break
        current = parent
    raise FileNotFoundError(
        f"向上 {max_depth} 层未找到 {marker}，请手动设置 PROJECT_ROOT。"
    )

PROJECT_ROOT = find_project_root()
print(f"项目根目录: {PROJECT_ROOT}")

# ---------- Python 环境 ----------
print(f"Python: {sys.version}")
print(f"Python 路径: {sys.executable}")

# ---------- CUDA 检测 ----------
try:
    import torch
    cuda_available = torch.cuda.is_available()
    if cuda_available:
        print(f"CUDA: 可用 (设备: {torch.cuda.get_device_name(0)})")
    else:
        print("CUDA: 不可用，将使用 CPU 训练")
except ImportError:
    cuda_available = False
    print("[警告] 未安装 PyTorch，无法检测 CUDA")

# ---------- ultralytics 检测 ----------
try:
    import ultralytics
    print(f"ultralytics: {ultralytics.__version__}")
except ImportError:
    print("[错误] 未安装 ultralytics，请在 yolov8 conda 环境中运行:")
    print("  conda activate yolov8")
    raise

## 2. 训练配置

修改下方变量即可调整训练参数，无需改动其他单元格。

In [ ]:
# ==================== 训练超参数 ====================
# 修改这里的值即可调整训练配置

EPOCHS: int = 120           # 训练轮数（建议 >= 120）
BATCH: int = 16             # batch size（4090 GPU 建议 16-32）
IMGSZ: int = 640            # 推理图像尺寸
DEVICE: str = "0" if cuda_available else "cpu"  # 自动选择 GPU/CPU
MODEL: str = "yolov8n.pt"   # 基础预训练模型
NAME: str = "unified_detect" # 训练运行名（影响输出目录名）

# ==================== 路径配置（自动推导） ====================
DEEPLEARN_DIR = PROJECT_ROOT / "deeplearn"
DATASET_YAML = DEEPLEARN_DIR / "merged_dataset" / "dataset.yaml"
OUTPUT_ROOT = DEEPLEARN_DIR / "runs" / "detect"

print(f"训练轮数: {EPOCHS}")
print(f"Batch size: {BATCH}")
print(f"图像尺寸: {IMGSZ}")
print(f"设备: {DEVICE}")
print(f"基础模型: {MODEL}")
print(f"运行名: {NAME}")
print(f"数据集配置: {DATASET_YAML}")
print(f"输出目录: {OUTPUT_ROOT / NAME}")

## 3. 数据集准备

自动检测合并数据集是否存在，不存在则调用 `prepare_merged_dataset.py` 生成。

In [ ]:
if DATASET_YAML.is_file():
    print(f"合并数据集已存在: {DATASET_YAML}")
else:
    print("合并数据集不存在，正在生成...")
    prepare_script = DEEPLEARN_DIR / "prepare_merged_dataset.py"
    if not prepare_script.is_file():
        raise FileNotFoundError(f"数据集准备脚本不存在: {prepare_script}")

    import subprocess
    result = subprocess.run(
        [sys.executable, str(prepare_script)],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f"[错误] 数据集生成失败:\n{result.stderr}")
        raise RuntimeError("数据集生成失败")
    print("合并数据集生成完成！")

## 4. 执行训练

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=str(DATASET_YAML),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    device=DEVICE,
    name=NAME,
    project=str(OUTPUT_ROOT),
)

print("\n训练完成！")

## 5. 结果回顾

In [ ]:
import csv
from IPython.display import Image, display

RUN_DIR = OUTPUT_ROOT / NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

print(f"best.pt 路径: {BEST_PT}")
print(f"best.pt 存在: {BEST_PT.is_file()}")

# 打印末轮指标
if RESULTS_CSV.is_file():
    with RESULTS_CSV.open("r", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    if rows:
        last = {k.strip(): v for k, v in rows[-1].items() if k}
        metrics = ["epoch", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]
        print("\n末轮关键指标:")
        for m in metrics:
            if m in last:
                print(f"  {m}: {last[m]}")

# 显示训练曲线
results_png = RUN_DIR / "results.png"
if results_png.is_file():
    print("\n训练曲线:")
    display(Image(filename=str(results_png)))

# 显示混淆矩阵
for cm_name in ["confusion_matrix.png", "confusion_matrix_normalized.png"]:
    cm_path = RUN_DIR / cm_name
    if cm_path.is_file():
        print(f"\n{cm_name}:")
        display(Image(filename=str(cm_path)))